In [1]:
import tensorflow as tf
import time

I0000 00:00:1790747970.159857    5751 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790747971.298988    5751 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790747974.592926    5751 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
tf.__version__

'2.21.0'

In [21]:
# read file in the disk and load it to the memory in the batches
class FileDataset(tf.data.Dataset):
    def read_file_in_bathes(num_sample):
        # open file
        time.sleep(0.03)
        for sample_idx in range(num_sample):
            time.sleep(0.015)
            yield (sample_idx,) #return index and we will can another one later
            
    def __new__(cls, num_sample=3):
        return tf.data.Dataset.from_generator( # take python generator and return it into tensorflow dataset
            cls.read_file_in_bathes,
            output_signature=tf.TensorSpec(shape = (1,), dtype = tf.int64),
            args=(num_sample,)
        )

In [22]:
# test how long it will take to process this dataset
def benchmark(dataset, num_epochs=2):
    for epochs in range(num_epochs):
        for sample in dataset:
            time.sleep(0.01)

In [23]:
%%timeit
benchmark(FileDataset())

E0000 00:00:1790749425.389035    5751 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1790749425.520070    5751 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


255 ms ± 9.68 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
%%timeit
# .prefetch() prepare next data while current one is processing, 1 means keep one data in advance
benchmark(FileDataset().prefetch(1))

254 ms ± 8.79 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%%timeit
# let tensorflow to device how much data it want to prepare based 
benchmark(FileDataset().prefetch(tf.data.AUTOTUNE))

252 ms ± 8.22 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
dataset = tf.data.Dataset.range(5)
for d in dataset:
    print(d.numpy())

0
1
2
3
4


In [35]:
dataset = dataset.map(lambda x: x** 2)

In [41]:
dataset.cache()
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [42]:
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]